# 00. Dataset Generation (CUDA)

Multi-Chromatic Spatial Pattern 시뮬레이션 데이터 생성.

**환경**: Google Colab (A100 GPU)

**구성:**
1. CUDA Toolkit 설치
2. Co-culture Model CUDA 코드 작성 (`.cu`)
3. 컴파일 (`nvcc`)
4. 파라미터 스윕 실행 (512개 = 8×8×8)

**출력**: `ParamSweep_{1..512}_Output/` → `Pos_RR_RG_GG.dat`, `Types_RR_RG_GG.dat`

In [ ]:
# URP 2025
# Multi-Chromatic Spatial Pattern Classification using Chromatic Topological Data Analysis
# Hanjin Tak (20230802), Juyoung Jo (20230719) KAIST

## 1. CUDA Toolkit 설치

In [ ]:
!apt-get update
!apt-get install -y cuda-toolkit-11-8

## 2. Co-culture Model CUDA 코드

200개 세포 (Red 40%, Green 60%)의 상호작용 시뮬레이션.

**핵심 파라미터:**
- `RR, RG, GG`: 세포 간 접착 강도 (adhesion strength)
- `boxsize = 10.0`, `nbd_eps = 1.5`: 주기 경계 조건 (PBC)
- `end_time = 5,000,000`, `dt = 0.02`
- Lennard-Jones like force: `F = (cA/lA)*exp(-r/lA) - (cR/lR)*exp(-r/lR)`

**CUDA Kernels:**
- `initialize_gpu_data_kernel`: RNG 초기화, 타이머 초기화
- `calculate_forces_with_ghosts_kernel`: 직접 이웃 + phantom(ghost) 이웃 힘 계산 (PBC)
- `update_positions_and_timers_kernel`: 위치 업데이트, PBC wrap, 재편극화

In [ ]:
%%writefile coculture_model_cuda_fin.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <string.h>
#include <time.h>
#include <assert.h>

// CUDA specific headers
#include <cuda_runtime.h>
#include <cuComplex.h>
#include <curand_kernel.h>

#define MAX_FILENAME 512

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            fprintf(stderr, "CUDA Error at %s:%d - %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

// Function prototypes
void coculture_model(int task_id, double RR, double RG, double GG);
int randsample(int n, double *weights);
cuDoubleComplex *get_init_pos_cpu_only(double limit, int n);
cuDoubleComplex *get_unit_dir_cpu_only(int n);
void write_complex_array_host(const char *filename, cuDoubleComplex *arr, int n);
void write_int_array_host(const char *filename, int *arr, int n);

// ====================== CUDA Kernels ======================

__global__ void initialize_gpu_data_kernel(
    int *p_timer, int *mitosis_timer, curandState_t *rand_states, int n,
    int cell_polarity_offset, int cell_cycle_offset, unsigned long long seed
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        curand_init(seed, idx, 0, &rand_states[idx]);
        p_timer[idx]     = (int)(curand_uniform_double(&rand_states[idx]) * (cell_polarity_offset + 1));
        mitosis_timer[idx]= (int)(curand_uniform_double(&rand_states[idx]) * (cell_cycle_offset   + 1));
    }
}

/*
 * Force kernel with PBC + phantom(ghost) neighbours:
 * 1) Direct neighbours: r = |z_k - z_j| < nbd_eps
 * 2) Phantom neighbours for PBC wrap
 */
__global__ void calculate_forces_with_ghosts_kernel(
    const cuDoubleComplex *d_z, const cuDoubleComplex *d_p, cuDoubleComplex *d_dz,
    const int *d_cell_types, const double *d_adhesion_strength,
    int n, double nbd_eps, int toggle_polarity,
    double lA, double lR, double boxsize
) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= n) return;

    const double xmin = -boxsize, xmax =  boxsize;
    const double ymin = -boxsize, ymax =  boxsize;

    cuDoubleComplex dz = make_cuDoubleComplex(0.0, 0.0);
    if (toggle_polarity) { dz = cuCadd(dz, d_p[j]); }
    cuDoubleComplex LJ_vec = make_cuDoubleComplex(0.0, 0.0);

    const double xj = cuCreal(d_z[j]);
    const double yj = cuCimag(d_z[j]);

    // (1) Direct neighbours
    for (int k = 0; k < n; ++k) {
        if (k == j) continue;
        const double dx = cuCreal(d_z[k]) - xj;
        const double dy = cuCimag(d_z[k]) - yj;
        const double r  = sqrt(dx*dx + dy*dy);
        if (r < nbd_eps && r > 0.0) {
            const int ctype = d_cell_types[j] - 1;
            const int ntype = d_cell_types[k] - 1;
            const double adh = d_adhesion_strength[ctype * 2 + ntype];
            const double cA  = 1.0 * adh;
            const double cR  = 0.25 * adh;
            const double F   = (cA/lA) * exp(-r/lA) - (cR/lR) * exp(-r/lR);
            cuDoubleComplex u = make_cuDoubleComplex(dx/r, dy/r);
            LJ_vec = cuCadd(LJ_vec, cuCmul(make_cuDoubleComplex(F, 0.0), u));
        }
    }

    // (2) Phantom neighbours (PBC)
    double x_p = 0.0, y_p = 0.0;
    if (xj - xmin < nbd_eps) x_p = xj - xmin + xmax;
    if (yj - ymin < nbd_eps) y_p = yj - ymin + ymax;
    if (!(x_p == 0.0 && y_p == 0.0)) {
        if (x_p > 0.0 && y_p == 0.0) y_p = yj;
        else if (x_p == 0.0 && y_p > 0.0) x_p = xj;
        const double px = x_p, py = y_p;
        for (int k = 0; k < n; ++k) {
            if (k == j) continue;
            const double dxp = cuCreal(d_z[k]) - px;
            const double dyp = cuCimag(d_z[k]) - py;
            const double rp  = sqrt(dxp*dxp + dyp*dyp);
            if (rp < nbd_eps && rp > 0.0) {
                const int ctype = d_cell_types[j] - 1;
                const int ntype = d_cell_types[k] - 1;
                const double adh = d_adhesion_strength[ctype * 2 + ntype];
                const double cA  = 1.0 * adh;
                const double cR  = 0.25 * adh;
                const double F   = (cA/lA) * exp(-rp/lA) - (cR/lR) * exp(-rp/lR);
                cuDoubleComplex u = make_cuDoubleComplex(dxp/rp, dyp/rp);
                LJ_vec = cuCadd(LJ_vec, cuCmul(make_cuDoubleComplex(F, 0.0), u));
            }
        }
    }
    d_dz[j] = cuCadd(dz, LJ_vec);
}

__global__ void update_positions_and_timers_kernel(
    cuDoubleComplex *d_z, const cuDoubleComplex *d_dz, int *d_p_timer,
    int *d_mitosis_timer, int n, double dt, double boxsize,
    int toggle_periodic_bdy, int toggle_cell_cycle,
    int polarity_duration, int /*cell_cycle_duration*/,
    cuDoubleComplex *d_p, curandState_t *rand_states, double pol_val
) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= n) return;
    d_z[j] = cuCadd(d_z[j], cuCmul(d_dz[j], make_cuDoubleComplex(dt, 0.0)));
    if (toggle_periodic_bdy) {
        double x = cuCreal(d_z[j]), y = cuCimag(d_z[j]);
        if (x >  boxsize) x -= 2.0 * boxsize;
        else if (x < -boxsize) x += 2.0 * boxsize;
        if (y >  boxsize) y -= 2.0 * boxsize;
        else if (y < -boxsize) y += 2.0 * boxsize;
        d_z[j] = make_cuDoubleComplex(x, y);
    }
    d_p_timer[j]++;
    if (d_p_timer[j] >= polarity_duration) {
        double theta = 2.0 * M_PI * curand_uniform_double(&rand_states[j]);
        d_p[j] = cuCmul(make_cuDoubleComplex(pol_val, 0.0),
                        make_cuDoubleComplex(cos(theta), sin(theta)));
        d_p_timer[j] = 0;
    }
    if (toggle_cell_cycle) d_mitosis_timer[j]++;
}

// ---------------------- Main Simulation ----------------------

void coculture_model(int task_id, double RR, double RG, double GG) {
    clock_t start_time = clock();
    int n = 200;
    double pol_val = 0.005;
    double cell_pop_prop[2] = {0.4, 0.6};
    int num_cell_types = 2;
    double adhesion_strength_cpu[2][2] = {{RR, RG}, {RG, GG}};
    double boxsize = 10.0, nbd_eps = 1.5;
    int cell_cycle_duration = 80000, polarity_duration = 2500;
    int cell_cycle_offset = (int)(0.8 * cell_cycle_duration);
    int cell_polarity_offset = (int)(0.8 * polarity_duration);
    int toggle_periodic_bdy = 1, toggle_polarity = 1, toggle_cell_cycle = 0;
    int end_time = 5000000;
    double dt = 0.02, lA = 14.0, lR = 0.5;

    char cond_str[MAX_FILENAME];
    snprintf(cond_str, MAX_FILENAME, "/content/drive/MyDrive/URP/ParamSweep_%d_Output", task_id);
    char mkdir_cmd[MAX_FILENAME + 32];
    snprintf(mkdir_cmd, sizeof(mkdir_cmd), "mkdir -p %s", cond_str);
    system(mkdir_cmd);

    srand(task_id);
    unsigned long long kernel_seed = (unsigned long long)time(NULL) + (unsigned long long)task_id;

    cuDoubleComplex *h_z = get_init_pos_cpu_only(boxsize, n);
    cuDoubleComplex *h_p_initial = get_unit_dir_cpu_only(n);
    for (int i = 0; i < n; i++)
        h_p_initial[i] = cuCmul(h_p_initial[i], make_cuDoubleComplex(pol_val, 0.0));

    int *h_cell_types = (int *)malloc(n * sizeof(int));
    for (int i = 0; i < n; i++)
        h_cell_types[i] = randsample(num_cell_types, cell_pop_prop) + 1;

    cuDoubleComplex *d_z, *d_p, *d_dz;
    int *d_p_timer, *d_mitosis_timer, *d_cell_types;
    double *d_adhesion_strength;
    curandState_t *d_rand_states;

    CUDA_CHECK(cudaMalloc((void**)&d_z, n * sizeof(cuDoubleComplex)));
    CUDA_CHECK(cudaMalloc((void**)&d_p, n * sizeof(cuDoubleComplex)));
    CUDA_CHECK(cudaMalloc((void**)&d_dz, n * sizeof(cuDoubleComplex)));
    CUDA_CHECK(cudaMalloc((void**)&d_p_timer, n * sizeof(int)));
    CUDA_CHECK(cudaMalloc((void**)&d_mitosis_timer, n * sizeof(int)));
    CUDA_CHECK(cudaMalloc((void**)&d_cell_types, n * sizeof(int)));
    CUDA_CHECK(cudaMalloc((void**)&d_adhesion_strength, 4 * sizeof(double)));
    CUDA_CHECK(cudaMalloc((void**)&d_rand_states, n * sizeof(curandState_t)));

    CUDA_CHECK(cudaMemcpy(d_z, h_z, n * sizeof(cuDoubleComplex), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_p, h_p_initial, n * sizeof(cuDoubleComplex), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_cell_types, h_cell_types, n * sizeof(int), cudaMemcpyHostToDevice));
    double h_adh_flat[4] = {adhesion_strength_cpu[0][0], adhesion_strength_cpu[0][1],
                            adhesion_strength_cpu[1][0], adhesion_strength_cpu[1][1]};
    CUDA_CHECK(cudaMemcpy(d_adhesion_strength, h_adh_flat, 4 * sizeof(double), cudaMemcpyHostToDevice));

    { int bs=256, gs=(n+bs-1)/bs;
      initialize_gpu_data_kernel<<<gs,bs>>>(d_p_timer, d_mitosis_timer, d_rand_states, n,
                                            cell_polarity_offset, cell_cycle_offset, kernel_seed);
      CUDA_CHECK(cudaGetLastError()); CUDA_CHECK(cudaDeviceSynchronize()); }

    int itr = 0;
    const int tpb = 256, bpg = (n + tpb - 1) / tpb;
    cuDoubleComplex *h_z_out = (cuDoubleComplex *)malloc(n * sizeof(cuDoubleComplex));
    int *h_ct_out = (int *)malloc(n * sizeof(int));

    while (itr <= end_time) {
        CUDA_CHECK(cudaMemset(d_dz, 0, n * sizeof(cuDoubleComplex)));
        calculate_forces_with_ghosts_kernel<<<bpg,tpb>>>(d_z, d_p, d_dz, d_cell_types,
            d_adhesion_strength, n, nbd_eps, toggle_polarity, lA, lR, boxsize);
        CUDA_CHECK(cudaGetLastError());
        update_positions_and_timers_kernel<<<bpg,tpb>>>(d_z, d_dz, d_p_timer, d_mitosis_timer,
            n, dt, boxsize, toggle_periodic_bdy, toggle_cell_cycle, polarity_duration,
            cell_cycle_duration, d_p, d_rand_states, pol_val);
        CUDA_CHECK(cudaGetLastError()); CUDA_CHECK(cudaDeviceSynchronize());
        if (itr == end_time) {
            char pf[MAX_FILENAME], tf[MAX_FILENAME];
            CUDA_CHECK(cudaMemcpy(h_z_out, d_z, n*sizeof(cuDoubleComplex), cudaMemcpyDeviceToHost));
            CUDA_CHECK(cudaMemcpy(h_ct_out, d_cell_types, n*sizeof(int), cudaMemcpyDeviceToHost));
            snprintf(pf, MAX_FILENAME, "%s/Pos_%.2f_%.2f_%.2f.dat", cond_str, RR, RG, GG);
            snprintf(tf, MAX_FILENAME, "%s/Types_%.2f_%.2f_%.2f.dat", cond_str, RR, RG, GG);
            write_complex_array_host(pf, h_z_out, n);
            write_int_array_host(tf, h_ct_out, n);
        }
        ++itr;
    }

    CUDA_CHECK(cudaFree(d_z)); CUDA_CHECK(cudaFree(d_p)); CUDA_CHECK(cudaFree(d_dz));
    CUDA_CHECK(cudaFree(d_p_timer)); CUDA_CHECK(cudaFree(d_mitosis_timer));
    CUDA_CHECK(cudaFree(d_cell_types)); CUDA_CHECK(cudaFree(d_adhesion_strength));
    CUDA_CHECK(cudaFree(d_rand_states));
    free(h_z); free(h_p_initial); free(h_cell_types); free(h_z_out); free(h_ct_out);
    printf("Total runtime: %.3f seconds\n", (double)(clock()-start_time)/CLOCKS_PER_SEC);
}

// -------------------- Host Helper Functions --------------------

cuDoubleComplex *get_init_pos_cpu_only(double limit, int n) {
    int cnt = 0;
    cuDoubleComplex *res = (cuDoubleComplex *)malloc(n * sizeof(cuDoubleComplex));
    while (cnt < n) {
        double r1 = -(limit-1.6) + 2*(limit-1.6)*((double)rand()/RAND_MAX);
        double r2 = -(limit-1.6) + 2*(limit-1.6)*((double)rand()/RAND_MAX);
        cuDoubleComplex pos = make_cuDoubleComplex(r1, r2);
        int too_close = 0;
        for (int j = 0; j < cnt; j++) {
            double dx = cuCreal(res[j])-cuCreal(pos), dy = cuCimag(res[j])-cuCimag(pos);
            if (sqrt(dx*dx+dy*dy) < 1.0) { too_close = 1; break; }
        }
        if (!too_close) res[cnt++] = pos;
    }
    return res;
}

cuDoubleComplex *get_unit_dir_cpu_only(int n) {
    cuDoubleComplex *res = (cuDoubleComplex *)malloc(n * sizeof(cuDoubleComplex));
    for (int i = 0; i < n; i++) {
        double theta = 2*M_PI*((double)rand()/RAND_MAX);
        res[i] = make_cuDoubleComplex(cos(theta), sin(theta));
    }
    return res;
}

int randsample(int n, double *weights) {
    double sum = 0.0;
    for (int i = 0; i < n; i++) sum += weights[i];
    double r = ((double)rand()/RAND_MAX)*sum, acc = 0.0;
    for (int i = 0; i < n; i++) { acc += weights[i]; if (r <= acc) return i; }
    return n - 1;
}

void write_complex_array_host(const char *fn, cuDoubleComplex *arr, int n) {
    FILE *f = fopen(fn, "w");
    for (int i = 0; i < n; i++) fprintf(f, "%lf,%lf\n", cuCreal(arr[i]), cuCimag(arr[i]));
    fclose(f);
}

void write_int_array_host(const char *fn, int *arr, int n) {
    FILE *f = fopen(fn, "w");
    for (int i = 0; i < n; i++) fprintf(f, "%d\n", arr[i]);
    fclose(f);
}

int main(int argc, char *argv[]) {
    if (argc < 5) { printf("Usage: %s <task_id> <RR> <RG> <GG>\n", argv[0]); return 1; }
    coculture_model(atoi(argv[1]), atof(argv[2]), atof(argv[3]), atof(argv[4]));
    return 0;
}

## 3. 컴파일

In [ ]:
!nvcc -std=c++11 -o coculture_cuda_fin coculture_model_cuda_fin.cu -lcudart -lm -arch=sm_80

## 4. Google Drive 마운트 & 파라미터 스윕 실행

8×8×8 = 512개 파라미터 조합 (RR, RG, GG ∈ {0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25})

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import subprocess
import os

A = [0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]

param_list = []
for x1 in A:
    for x2 in A:
        for x3 in A:
            param_list.append((x1, x2, x3))

EXECUTABLE = './coculture_cuda_fin'

# === 실행할 task_id 목록 지정 (전체: range(1, 513)) ===
task_id_list = list(range(1, 513))  # 전체 512개 실행
# task_id_list = [355, 363, 379, 427, 507]  # 또는 특정 번호만
# =================================

if not os.path.exists(EXECUTABLE):
    print(f"Error: Executable '{EXECUTABLE}' not found.")
else:
    for idx, (RR, RG, GG) in enumerate(param_list):
        task_id = idx + 1
        if task_id not in task_id_list:
            continue
        log_file = f'log_{task_id}.txt'
        cmd = [EXECUTABLE, str(task_id), str(RR), str(RG), str(GG)]
        print(f'Running: {cmd}')
        with open(log_file, 'w') as f:
            subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)

    print('Selected jobs completed.')